In [1]:
import pathlib
import pickle

import folium
import mcr_py.helper_functions
import mcr_py.mcr.data
import mcr_py.mcr.path
import mcr_py.mcr5.labels
import mcr_py.minute_city.minute_city
import mcr_py.utils.strtime
import numpy as np
import pandas as pd
import polars as pl
from mcr_py.mcr.path import GTFSPath, Path, PathType
from mcr_py.utils.logger import setup

setup("INFO")

In [2]:
city_name = "cologne"
date = "20250926"

In [3]:
data_directory = pathlib.Path("../data/")
base_directory = data_directory / date
cache_path = base_directory / "cache/"
osm_path = base_directory / "osm_raw"
geometa_path = base_directory / f"cache/{city_name}_geometa.json"
mcr5_output_path = base_directory / f"mcr5_results/{city_name}_reduced_paths"
mcr5_output_path_comp = base_directory / f"mcr5_results/{city_name}"
gtfs_clean_dir = base_directory / f"gtfs_clean/{city_name}/"
gtfs_clean_struct = gtfs_clean_dir / "structs.pkl"
gtfs_clean_stops = gtfs_clean_dir / "stops.parquet"
geo_meta, geo_data = mcr_py.helper_functions.load_auxiliary_classes(
    geo_meta_path=geometa_path,
    city_id="Koeln",
    osm_path=osm_path,
    cache_path=cache_path,
)

[10:27:59] INFO     Loading OSM walking                               ]8;id=729547;file:///Users/philipppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=655377;file:///Users/philipppeter/repo/mcr-py/python/mcr_py/mcr/data.py#63\63]8;;\
[10:28:01] INFO     Loading OSM walking done (1.79 seconds)           ]8;id=52135;file:///Users/philipppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=970888;file:///Users/philipppeter/repo/mcr-py/python/mcr_py/mcr/data.py#63\63]8;;\
           INFO     Loading OSM POIs                                  ]8;id=42270;file:///Users/philipppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=765755;file:///Users/philipppeter/repo/mcr-py/python/mcr_py/mcr/data.py#70\70]8;;\
           INFO     Loading OSM POIs done (0.01 seconds)              ]8;id=337432;file:///Users/philipppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=892338;file:///Users/philipppeter/repo/mcr-py/python/mcr

In [113]:
with open(mcr5_output_path / "public_transport_0" / "891fa199c77ffff.pkl", "rb") as f:
    hex = pickle.load(f)

In [114]:
comp = pl.read_ipc(mcr5_output_path_comp / "bicycle" / "891fa199c77ffff.feather").join(
    geo_data.pois.select("nearest_osm_node", "poi_type", "lat", "long"),
    how="left",
    left_on="osm_node_id",
    right_on="nearest_osm_node",
)

Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


In [115]:
labels = pd.DataFrame(
    [
        (label.node_id, label.values[0], label.values[1], n_transfers, label)
        for n_transfers, bags in hex["bags_i"].items()
        for bag in bags.values()
        for label in bag
    ],
    columns=["osm_node_id", "time", "cost", "n_transfers", "label"],
)

In [116]:
labels = labels.merge(
    geo_data.pois.select(
        pl.col("nearest_osm_node").alias("osm_node_id").cast(pl.Int64),
        pl.col("lat").alias("poi_lat"),
        pl.col("long").alias("poi_long"),
        "poi_type",
    ).to_pandas(),
    how="left",
    on="osm_node_id",
)

In [117]:
labels["duplicate"] = labels.duplicated(
    subset=["osm_node_id", "time", "cost", "poi_type", "poi_lat", "poi_long"], keep=False
)

In [118]:
labels[labels["duplicate"]].sort_values(by=["osm_node_id", "time"])

,osm_node_id,time,cost,n_transfers,label,poi_lat,poi_long,poi_type,duplicate


In [119]:
nodes = geo_data.osm_nodes.with_columns(pl.col("osm_id").alias("id")).to_pandas()

In [120]:
path_manager = hex["path_manager"]

In [121]:
from mcr_py.mcr.data import NetworkType

translator_map = {
    PathType.WALKING: dict(
        geo_data.osm_nodes.select(pl.col("rx_node_id").alias("osm"), "osm_id").rows()
    ),
    PathType.CYCLING_WALKING: dict(
        pl.concat(
            [
                geo_data.osm_nodes.select(
                    pl.lit("W").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
                geo_data.additional_networks[NetworkType.CYCLING][0].select(
                    pl.lit("D").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
            ],
            how="diagonal",
        )
        .with_row_index()
        .rows()
    ),
    PathType.DRIVING_WALKING: dict(
        pl.concat(
            [
                geo_data.osm_nodes.select(
                    pl.lit("W").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
                geo_data.additional_networks[NetworkType.DRIVING][0].select(
                    pl.lit("D").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
            ],
            how="diagonal",
        )
        .with_row_index()
        .rows()
    ),
    PathType.PUBLIC_TRANSPORT: None,
}

In [122]:
def format_meta(meta, previous_meta, start_time):
    values = meta["values"]
    arrival_time = values[0]
    cost = values[1]

    if previous_meta:
        previous_values = previous_meta["values"]
        previous_arrival_time = previous_values[0]
        previous_cost = previous_values[1]

        arrival_time -= previous_arrival_time
        cost -= previous_cost
    else:
        arrival_time -= start_time

    return f"{mcr_py.utils.strtime.seconds_to_str_time(arrival_time, 10)} ({cost})"

In [123]:
labels.poi_type.unique()

array([nan, 'Sustenance', 'Grocery', 'Banks', 'Shops', 'Health', 'Parks',
       'Education'], dtype=object)

In [124]:
color_map = {
    "Shops": "orange",
    "Grocery": "red",
    "Parks": "green",
    "Education": "blue",
    "Banks": "violet",
    "Health": "darkgreen",
    "Sustenance": "yellow",
}

mode_color = {
    PathType.WALKING: "grey",
    PathType.CYCLING_WALKING: "blue",
    PathType.DRIVING_WALKING: "violet",
    PathType.PUBLIC_TRANSPORT: "green",
}

In [127]:
MLC_PATH_TYPES = [
    PathType.WALKING.value,
    PathType.CYCLING_WALKING.value,
    PathType.DRIVING_WALKING.value,
    PathType.WALKING,
    PathType.CYCLING_WALKING,
    PathType.DRIVING_WALKING,
]


def alt_reconstruct_and_translate_path_for_label(paths, label, translator_map) -> list:
    translated_path = []
    for path_id in label.path:
        assert isinstance(path_id, int)
        path = paths[path_id]
        if path.path_type in MLC_PATH_TYPES:
            translated_path.append(
                Path(
                    path_type=path.path_type,
                    path=[translator_map[path.path_type][p] for p in path.path],
                    meta=path.meta,
                )
            )
        elif path.path_type == PathType.PUBLIC_TRANSPORT:
            translated_path.append(
                GTFSPath(
                    start_stop_id=int(path.path[0]),
                    # trip_id=str(path.path[1]),
                    trip_id="",
                    end_stop_id=int(path.path[1]),
                    meta=path.meta,
                )
            )
        else:
            msg = f"Unknown path type {path.path_type}"
            raise ValueError(msg)
    return translated_path

In [132]:
import plotly.graph_objects as go
from mcr_py.mcr.label import IntermediateLabel


def plot_paths_on_map(
    labels: pd.DataFrame,
    nodes: pd.DataFrame,
    path_manager: mcr_py.mcr.path.PathManager,
    translator_map: dict[PathType, dict[str, int]],
    color_map: dict[str, str],
    path_name: str,
) -> None:
    nodes_by_id = nodes.set_index("id", drop=False)
    fig = go.Figure()

    walking_paths_long = []
    walking_paths_lat = []
    cycling_paths_long = []
    cycling_paths_lat = []
    poi_assoc_long = []
    poi_assoc_lat = []

    for row in labels.itertuples():
        label: IntermediateLabel = row.label  # pyright: ignore[reportAssignmentType]
        end_node_id = row.osm_node_id
        end_node = nodes_by_id.loc[end_node_id]

        # paths = mcr_py.mcr.path.reconstruct_and_translate_path_for_label(
        #     path_manager.paths, label, translator_map
        # )
        paths = alt_reconstruct_and_translate_path_for_label(
            path_manager.paths, label, translator_map
        )
        for i, path in enumerate(paths):
            if isinstance(path, Path):
                if path.path == []:
                    continue
                if path.path_type == PathType.WALKING:
                    walking_path_nodes = [nodes_by_id.loc[node_id] for node_id in path.path]
                    path_lat = [node.lat for node in walking_path_nodes]
                    path_lon = [node.long for node in walking_path_nodes]
                    if i + 1 == len(paths):
                        path_lat.append(end_node.lat)  # type: ignore
                        path_lon.append(end_node.long)  # type: ignore
                    else:
                        pass
                        # path_lat.append(nodes_by_id.loc[int(paths[i + 1].path[0][1:])].lat)
                        # path_lon.append(nodes_by_id.loc[int(paths[i + 1].path[0][1:])].long)
                    walking_paths_long.extend(path_lon)
                    walking_paths_long.append(None)
                    walking_paths_lat.extend(path_lat)
                    walking_paths_lat.append(None)
                if path.path_type in [PathType.DRIVING_WALKING, PathType.CYCLING_WALKING]:
                    path_nodes = [
                        nodes_by_id.loc[int(node_id[1:])]  # pyright: ignore[reportIndexIssue]
                        for node_id in path.path
                        if node_id[0] == "D"  # pyright: ignore[reportIndexIssue]
                    ]
                    path_lat = [node.lat for node in path_nodes]
                    path_lon = [node.long for node in path_nodes]
                    cycling_paths_long.extend(path_lon)
                    cycling_paths_long.append(None)
                    cycling_paths_lat.extend(path_lat)
                    cycling_paths_lat.append(None)
        if row.poi_type is not np.nan:
            poi_assoc_long.append(row.poi_long)
            poi_assoc_long.append(end_node.long)
            poi_assoc_lat.append(row.poi_lat)
            poi_assoc_lat.append(end_node.lat)
            poi_assoc_long.append(None)
            poi_assoc_lat.append(None)

    fig.add_trace(
        go.Scattermap(
            lon=walking_paths_long,
            lat=walking_paths_lat,
            mode="lines",
            marker={"size": 1, "color": "grey"},
            legendgroup="Walking",
            name="Walking",
            showlegend=True,
        )
    )
    fig.add_trace(
        go.Scattermap(
            lon=cycling_paths_long,
            lat=cycling_paths_lat,
            mode="lines",
            marker={"size": 1, "color": "blue"},
            legendgroup="Bicycle",
            name="Bicycle",
            showlegend=True,
        )
    )

    nodes_long = [nodes_by_id.loc[row.osm_node_id].long for row in labels.itertuples()]
    nodes_lat = [nodes_by_id.loc[row.osm_node_id].lat for row in labels.itertuples()]
    fig.add_trace(
        go.Scattermap(
            lon=nodes_long,
            lat=nodes_lat,
            mode="markers",
            marker={"size": 8, "color": "grey"},
            name="Node",
            legendgroup="Nodes",
            showlegend=True,
        )
    )

    fig.add_trace(
        go.Scattermap(
            lon=poi_assoc_long,
            lat=poi_assoc_lat,
            mode="lines",
            marker={"size": 8, "color": "grey"},  # pyright: ignore[reportArgumentType]
            showlegend=False,
        )
    )

    for poi_type in labels.poi_type.dropna().unique():
        labels_with_poi_type = labels[labels["poi_type"] == poi_type]
        poi_long = [
            row.poi_long
            for row in labels_with_poi_type.itertuples()
            if row.poi_type == poi_type
        ]
        poi_lat = [
            row.poi_lat
            for row in labels_with_poi_type.itertuples()
            if row.poi_type == poi_type
        ]
        fig.add_trace(
            go.Scattermap(
                lon=poi_long,
                lat=poi_lat,
                mode="markers",
                marker={"size": 10, "color": color_map[poi_type]},
                name=poi_type,
                legendgroup=poi_type,
                showlegend=True,
            )
        )

    fig.add_trace(
        go.Scattermap(
            lat=["50.948884"],  # Latitude of the marker
            lon=["6.917342"],  # Longitude of the marker
            mode="markers",
            marker={"size": 14, "color": "lightgreen"},
            text=["Starting Point"],  # Hover text
            name="Starting Point",
            showlegend=True,
            legendgroup="Starting Point",
        )
    )

    fig.update_layout(
        map={
            "style": "carto-positron-nolabels",
            "zoom": 16,  # street level
            "center": {"lat": 50.948884, "lon": 6.917342},
        },
        legend={
            "title": {
                "text": "Transport & POI Types",  # your legend title
                "font": {"size": 14, "color": "black"},
            },
            "orientation": "v",  # vertical (default) or "h" for horizontal
            "x": 1,  # horizontal position (0=left, 1=right)
            "y": 1,  # vertical position (0=bottom, 1=top)
        },
        margin={"r": 0, "t": 0, "l": 0, "b": 0},
        height=2000,
        width=2400,
    )
    fig.write_image(path_name, scale=2)
    fig.show("firefox")

In [133]:
plot_paths_on_map(
    labels=labels,
    nodes=nodes,
    path_manager=path_manager,
    translator_map=translator_map,
    color_map=color_map,
    path_name="../figures/mcr5/cologne_reduced_20250926/public_transport_ehrenfeld.png",
)

[11:11:17] INFO     Chromium init'ed with kwargs {}              ]8;id=731374;file:///Users/philipppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/browsers/chromium.py\chromium.py]8;;\:]8;id=223673;file:///Users/philipppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/browsers/chromium.py#182\182]8;;\
           INFO     Found chromium path: /Applications/Google    ]8;id=989811;file:///Users/philipppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/browsers/chromium.py\chromium.py]8;;\:]8;id=861473;file:///Users/philipppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/browsers/chromium.py#209\209]8;;\
                    Chrome.app/Contents/MacOS/Google Chrome                     
           INFO     Temp directory created:                       ]8;id=107576;file:///Users/philipppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/utils/_tmpfile.py\_tmpfile.py]8;;\:]8;id=694259;file:

In [ ]:
toloop = labels

# stops_by_id = stops_df.set_index("stop_id")
sample_label = labels.iloc[0]
sample_node_id = sample_label.osm_node_id
nodes_by_id = nodes.set_index("id", drop=False)
sample_node = nodes_by_id.loc[sample_node_id]
start_time = 288000

m = folium.Map(location=[50.948884, 6.917342], zoom_start=17)


for row in toloop.itertuples():
    label: IntermediateLabel = row.label  # pyright: ignore[reportAssignmentType]
    end_node_id = row.osm_node_id
    end_node = nodes_by_id.loc[end_node_id]

    paths = mcr_py.mcr.path.reconstruct_and_translate_path_for_label(
        path_manager.paths, label, translator_map
    )
    for i, path in enumerate(paths):
        if isinstance(path, Path):
            if path.path == []:
                continue
            if path.path_type == PathType.WALKING:
                walking_path_nodes = [nodes_by_id.loc[node_id] for node_id in path.path]
                path_lat_lon = [(node.lat, node.long) for node in walking_path_nodes]
                if i + 1 == len(paths):
                    path_lat_lon.append([end_node.lat, end_node.long])
                else:
                    path_lat_lon.append(
                        (
                            nodes_by_id.loc[int(paths[i + 1].path[0][1:])].lat,
                            nodes_by_id.loc[int(paths[i + 1].path[0][1:])].long,
                        )
                    )
                last_node = path_lat_lon[-1]

                previous_meta = paths[i - 1].meta if i > 0 else None
                meta = format_meta(path.meta, previous_meta, start_time)
                if path_lat_lon != []:
                    folium.PolyLine(
                        [*path_lat_lon],
                        color="grey",
                        weight=2,
                        popup=str(meta),
                    ).add_to(m)
            if path.path_type in [PathType.DRIVING_WALKING, PathType.CYCLING_WALKING]:
                path_nodes = [
                    nodes_by_id.loc[int(node_id[1:])]
                    for node_id in path.path
                    if node_id[0] == "D"
                ]
                path_lat_lon = [(node.lat, node.long) for node in path_nodes]
                previous_meta = paths[i - 1].meta if i > 0 else None
                meta = format_meta(path.meta, previous_meta, start_time)
                if path_lat_lon != []:
                    folium.PolyLine(
                        path_lat_lon,
                        color=mode_color[path.path_type],
                        weight=2,
                        popup=str(meta),
                        dash_array=10,
                    ).add_to(m)
        elif isinstance(path, GTFSPath):
            print("Impossible")
            start_stop_id = path.start_stop_id
            end_stop_id = path.end_stop_id
            start_stop = stops_by_id.loc[start_stop_id]
            end_stop = stops_by_id.loc[end_stop_id]
            trip = path.trip_id
            if len(trip) >= 10:
                trip = trip[:10] + "..."

            previous_meta = paths[i - 1].meta if i > 0 else None
            line_msg = f"Trip: {trip}\n---\n {format_meta(path.meta, previous_meta)}"

            path_lat_lon = [
                (float(start_stop.stop_lat), float(start_stop.stop_lon)),
                (float(end_stop.stop_lat), float(end_stop.stop_lon)),
            ]
            folium.PolyLine(
                path_lat_lon,
                color="green",
                weight=2,
                popup=line_msg,
            ).add_to(m)

            folium.CircleMarker(
                location=[float(start_stop.stop_lat), float(start_stop.stop_lon)],
                popup=f"Start: {start_stop.stop_name}",
                color="green",
                radius=3,
            ).add_to(m)
            folium.CircleMarker(
                location=[float(end_stop.stop_lat), float(end_stop.stop_lon)],
                popup=f"End: {end_stop.stop_name}",
                color="green",
                radius=3,
            ).add_to(m)
        else:
            raise Exception("Unknown path type")

    folium.CircleMarker(
        location=[end_node.lat, end_node.long],  # pyright: ignore[reportArgumentType]
        popup=f"End: {end_node_id}",
        color="black",
        radius=2,
    ).add_to(m)
    if row.poi_type is not np.nan:
        folium.CircleMarker(
            location=[row.poi_lat, row.poi_long],  # pyright: ignore[reportArgumentType]
            popup=f"POI: {row.poi_type}",
            color=color_map[row.poi_type],
            radius=4,
        ).add_to(m)
        folium.PolyLine(
            [(row.poi_lat, row.poi_long), (end_node.lat, end_node.long)],
            color=color_map[row.poi_type],
            weight=2,
            popup=f"POI: {row.poi_type}",
        ).add_to(m)

folium.Marker(location=[50.948884, 6.917342], icon=folium.Icon("green"), popup="Start").add_to(
    m
)
m

In [ ]:
for path in path_manager.paths.values():
    if path.path_type == PathType.PUBLIC_TRANSPORT:
        print(path)

In [ ]:
for path in path_manager.paths.values():
    if path.path_type == PathType.PUBLIC_TRANSPORT:
        print(path)